In [1]:
!pip install -q langchain==0.3.24 langgraph==0.3.33 langgraph-supervisor langchain_google_genai langchain-community qdrant_client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 

In [2]:
!gdown 1rdpc-8KjZhjpRqBbZoFlQ6PYfobSwGiX
!unzip image_base64.zip

Downloading...
From: https://drive.google.com/uc?id=1rdpc-8KjZhjpRqBbZoFlQ6PYfobSwGiX
To: /content/image_base64.zip
100% 26.2k/26.2k [00:00<00:00, 51.5MB/s]
Archive:  image_base64.zip
  inflating: image_base64.txt        


In [4]:
import os
import base64
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, HarmBlockThreshold, HarmCategory
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

In [34]:
# Get API Gemini: https://aistudio.google.com/apikey
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=1,
    google_api_key=" ***your_api_gemini*** ",
    safety_settings={
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
    }
)

embedding_model = SentenceTransformer('sentence-transformers/sentence-t5-base')

qdrant_client = QdrantClient(
    url=" ***your_qdrant_url*** ",
    api_key=" ***your_api_key*** ",
)

In [50]:
@tool
def query_gemini_with_image_and_text_tool(base64_file_path: str, user_text_query: str) -> str:
    """
    Gửi ảnh (dạng base64 trong file) và câu hỏi để Gemini phân tích hình ảnh hoặc nhận diện vật thể.

    Args:
        base64_file_path: Đường dẫn file chứa chuỗi ảnh base64.
        user_text_query: Câu hỏi liên quan đến ảnh (ví dụ: "Hình này có gì?").

    Returns:
        Kết quả trả về từ Gemini dưới dạng mô tả hoặc JSON, hoặc thông báo lỗi nếu có sự cố.
    """
    try:
        with open(base64_file_path, "r") as f:
            base64_str = f.read()

        message = HumanMessage(
            content=[
                {"type": "text", "text": user_text_query},
                {"type": "image_url", "image_url": f"data:image/jpeg;base64,{base64_str}"},
            ]
        )
        response = llm.invoke([message])

        return response.content
    except FileNotFoundError:
        return f"Lỗi: Không tìm thấy file '{base64_file_path}'."
    except Exception as e:
        return f"Lỗi khi gọi Gemini với hình ảnh: {e}"


@tool
def retrieve_from_qdrant(query: str, top_k: int = 3) -> str:
    """
    Tìm kiếm và trả về top K đoạn mô tả liên quan trong bộ sưu tập Qdrant dựa trên câu hỏi.

    Args:
        query: Câu hỏi hoặc từ khóa tìm kiếm.
        top_k: Số lượng kết quả trả về (mặc định 3).

    Returns:
        Chuỗi chứa các đoạn mô tả phù hợp, ngăn cách bởi 2 dòng trống,
        hoặc thông báo lỗi nếu có sự cố.
    """
    try:
        embedding = embedding_model.encode(query, convert_to_tensor=False).tolist()

        results = qdrant_client.search(
            collection_name="my_store",
            query_vector=embedding,
            limit=top_k,
        )
        texts = [hit.payload.get("original_text", "Không có mô tả.") for hit in results]
        return "\n\n".join(texts)
    except Exception as e:
        return f"Lỗi khi truy xuất Qdrant: {e}"

In [51]:
system_instruction_text = """
Bạn là một trợ lý thông minh được thiết kế để hỗ trợ khách hàng các câu hỏi liên quan đến sản phẩm của công ty ABCTech. Bạn có quyền truy cập vào hai công cụ mạnh mẽ:
1.  `query_gemini_with_image_and_text_tool`:
    * Sử dụng công cụ này khi khách hàng có cung cấp một đường dẫn đến file chứa chuỗi Base64 của hình ảnh (ví dụ: 'image_abc.txt').
2.  `retrieve_from_qdrant`:
    * Sử dụng công cụ này khi khách hàng hỏi về thông tin sản phẩm, hoặc bất kỳ dữ liệu nào liên quan đến các sản phẩm của công ty.
Bạn chỉ được cung cấp các câu trả lời về thông tin sản phẩm của công ty dựa vào thông tin trả về của tool `retrieve_from_qdrant`. Nếu bạn không thể tìm thấy thông tin phù hợp, hãy thông báo cho khách hàng một cách lịch sự.
"""

agent_executor = create_react_agent(
    model=llm,
    tools=[query_gemini_with_image_and_text_tool, retrieve_from_qdrant],
    prompt=system_instruction_text
)

In [52]:
def run_agent_with_verbose(agent_executor, query):
    result = agent_executor.invoke(
    {"messages": [{"role": "user", "content": query}]}
    )
    messages = result["messages"]

    print("\n=== Bắt đầu quá trình agent xử lý ===\n")

    for i, msg in enumerate(messages):
        msg_type = type(msg).__name__

        print(f"Step {i+1}: {msg_type}")

        if isinstance(msg, HumanMessage):
            print(f"  Nội dung câu hỏi / input từ người dùng:\n  {msg.content}\n")

        elif isinstance(msg, AIMessage):
            print(f"  Phản hồi AI / Agent:")
            print(f"  Nội dung: {msg.content}\n")

            if 'function_call' in msg.additional_kwargs:
                fc = msg.additional_kwargs['function_call']
                print(f"  -> Gọi tool / hàm: {fc.get('name')}")
                print(f"  -> Tham số: {fc.get('arguments')}\n")

        elif isinstance(msg, ToolMessage):
            print(f"  Kết quả trả về từ tool '{msg.name}':")
            print(f"  {msg.content}\n")

        else:
            print(f"  Nội dung message:\n  {msg}\n")

    print("=== Kết thúc quá trình agent xử lý ===\n")

    return result

In [53]:
combined_query = "Tôi đang cần tìm sản phẩm như trong ảnh 'image_base64.txt', cửa hàng bạn có sản phẩm này không, nếu có hãy cho tôi biết thông tin và giá của sản phẩm đó?"

result = run_agent_with_verbose(agent_executor, combined_query)

/tmp/ipython-input-50-1940058272.py:48: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(



=== Bắt đầu quá trình agent xử lý ===

Step 1: HumanMessage
  Nội dung câu hỏi / input từ người dùng:
  Tôi đang cần tìm sản phẩm như trong ảnh 'image_base64.txt', cửa hàng bạn có sản phẩm này không, nếu có hãy cho tôi biết thông tin và giá của sản phẩm đó?

Step 2: AIMessage
  Phản hồi AI / Agent:
  Nội dung: 

  -> Gọi tool / hàm: query_gemini_with_image_and_text_tool
  -> Tham số: {"base64_file_path": "image_base64.txt", "user_text_query": "S\u1ea3n ph\u1ea9m trong \u1ea3nh l\u00e0 g\u00ec?"}

Step 3: ToolMessage
  Kết quả trả về từ tool 'query_gemini_with_image_and_text_tool':
  Sản phẩm trong ảnh là một chiếc iPhone 13 Pro Max.

Step 4: AIMessage
  Phản hồi AI / Agent:
  Nội dung: Bạn đang tìm iPhone 13 Pro Max phải không ạ? Để tôi kiểm tra thông tin và giá của sản phẩm này nhé.

  -> Gọi tool / hàm: retrieve_from_qdrant
  -> Tham số: {"query": "iPhone 13 Pro Max", "top_k": 5.0}

Step 5: ToolMessage
  Kết quả trả về từ tool 'retrieve_from_qdrant':
  Màn hình ProMotion OLED 6.9 in